####2. Convert strings to date

1. Spark validates the date against the Proleptic Gregorian calendar.
2. The negative years are BC, and the positive values are AD in Gregorian calander.
3. Valid dates are taken, and invalid dates throw an exception or taken as null

In [0]:
from pyspark.sql.functions import concat_ws

data_list = [(2022, 5, 18) , (9999, 12, 31), (-9999, 1, 1), (10000, 1, 1), 
             (-10000, 1, 1), (193, 5, 25),   (99, 5, 25),   (1000, 2, 29)]

df = (spark.createDataFrame(data_list).toDF("Y", "M", "D")
     .withColumn("date_str", concat_ws("-", "Y", "M", "D")))

df.display()


In [0]:
df.printSchema()

In [0]:
from pyspark.sql.functions import expr
date = (
    df.withColumn("valid_date", expr("try_to_date(date_str, 'y-M-d')"))
        .drop("Y", "M", "D")
    )


In [0]:
date.printSchema()

In [0]:
date.display()

In [0]:
from pyspark.sql.functions import date_add, date_sub, add_months

df_1 = date.withColumns(
    {
        "add_5_days": date_add("valid_date", 5),
        "sub_5_days": date_sub("valid_date", 5),
        "add_5_months": add_months("valid_date", 5),
        "sun_5_months": add_months("valid_date",-5)
    }
)
df_1.display()

In [0]:
from pyspark.sql.functions import current_date, date_diff, col
df = (
    df_1.withColumns({
        "current_date": current_date(),
        "delta_date_days": date_diff("add_5_months", "valid_date"),
        "delta_date_interval": col("current_date") - col("valid_date")
    })
)
df.display()

##### format datatype

In [0]:
from pyspark.sql.functions import date_format

df = (
    df.withColumn("fmt_date", date_format("valid_date", "dd MMM yyyy"))
)

df.display()